<font size = '6'>
DISPLAYING GEOSPATIAL MAP WITH PLOTLY AND PYDECK

In [ ]:
#Let's create spatial map with Plotly
import plotly.express as px
fig=px.choropleth(locations=["USA", "CAN", "MEX", "ARG", "BRA"], locationmode="ISO-3")
fig.update_layout(geo=dict(showcountries=True)) #Enable country boundaries
fig.show() #let's show the map

In [ ]:
fig.write_html("plotly_map.html")  # Save the map as an HTML file

In [ ]:
#Let's display a map using pydeck
import pydeck as pdk
view=pdk.ViewState(latitude=0, longitude=0, zoom=0.5)
layer=pdk.Layer("ScatterplotLayer", data=[])
deck=pdk.Deck(layers=[layer], initial_view_state=view)
deck.show()

In [ ]:
deck.to_html("pydeck_map.html") # Save the map as an HTML file. pydeck uses to_html() method to save the map as an HTML file.

<font size="6">
Calculating Distance Between two locations

In [ ]:
#Import the necessary packages
import geopandas as gpd
from geopy.distance import geodesic

#Create a dictionary to store the location data
data={
    "city":["Toledo", "Columbus"],
    "latitude": [41.651031, 39.9612],
    "longitude":[-83.541939, -82.9988]
}

#Create a geodataframe from the dictionary
gdf=gpd.GeoDataFrame(data, geometry=gpd.points_from_xy(data["longitude"], data["latitude"]))

#Extract the coordinates from each city
Coord_Toledo=(gdf.loc[0, "latitude"],gdf.loc[0, "longitude"])
Coord_Columbus=(gdf.loc[1, "latitude"],gdf.loc[1, "longitude"])

#Now let's calculte the distance between the locations.
distance=geodesic(Coord_Toledo,Coord_Columbus).miles
distance2=geodesic(Coord_Toledo,Coord_Columbus).kilometers
print (f"The distance between Toledo and Columbus is {distance:.2f} miles or {distance2:.2f} kilometers.")


In [ ]:
#Now let's visualize these locations using folium
import folium as fl
#first let's get the mid-point between Toledo and Columbus
midpoint=[(gdf["latitude"].mean()), (gdf["longitude"]).mean()]
#let's create folium map and set the location to midpoint
m=fl.Map(location=midpoint, zoom_start=3)
#we will add markers to the map to be able to see the points locations
for idx, row in gdf.iterrows():
    fl.Marker(location=[row["latitude"], row["longitude"]], popup=row["city"]).add_to(m)

#let's create a line to connect the two points
line=fl.PolyLine(locations=[(gdf.loc[0, "latitude"],gdf.loc[0, "longitude"]),
                            (gdf.loc[1, "latitude"],gdf.loc[1, "longitude"])],
                            color="red", weight=2.5, opacity=1)
#let's add this line we created to the map
m.add_child(line)
m

<font size="6">
Extracting Coordinates from Map

we need ipyleaflet and ipywidget to do this. We will install those first and import modules from them

In [ ]:
from ipywidgets import Label
from ipyleaflet import Map, Marker, Popup

#We will create a map using ipyleaflet and add a label to display the coordinates of the clicked location

m1 =Map(center=[0, 0], zoom=2)
label=Label()
display(label)

#we will create a function to handle map interaction
def handle_inter(**kwarges):
    if kwarges.get("type")=="click":
        lat, long=kwarges.get("coordinates")
        #we will store the coordinates in the label
        label.value=f"Clicked on:{lat:.5f}, {long:.5f}"
        #we will create a marker at the clicked location
        marker=Marker(location=(lat, long))
        #add marker to the map
        m1.add_layer(marker)

#now we will add the interaction function to the map
m1.on_interaction(handle_inter)
display(m1)




<font size="6">
Geocoding and Geocoding Reverse

In [ ]:
#FIRST WE WILL PERFORM GEOCODING TO GET THE COORDINATES OF A LOCATION
from geopy.geocoders import Nominatim

geolocator=Nominatim(user_agent="geocoder")
address="3106 Carskaddon Ave, Toledo, OH 43606, USA" #We will use address spreadsheet later

#Now let's perform geocoding
try:
    location=geolocator.geocode(address)
    #let check if the location is found
    if location:
        print(f"Address: {address}")
        print(f"Latitude: {location.latitude}, Longitude: {location.longitude}")
except Exception as e:
    print(f'Error: {e}')


In [ ]:
#NOW WE WILL PERFORM REVERSE GEOCODING TO GET THE ADDRESS OF A LOCATION
geolocator=Nominatim(user_agent="geocoder")

#We will get the coordinates of a location
latlong=[41.66141, -83.61622]
#longitude=-83.61622

#reverse geocoding
try:
    location=geolocator.reverse(latlong, exactly_one=True) #we are using exactly_one=True to get a single address
    #let's check if the location is found
    if location:
        print(f'Coordinates: {latlong}')
        print(f'Address: {location.address}')
    else:
        print(f'No address found for coordinates: {latlong}')
except Exception as e:
    print (f'Error: {e}')



<font size ="6">
Finding Nearby Cities Using Proximity Analysis

In [ ]:
from ipyleaflet import Map, Marker, CircleMarker, basemaps, LayersControl
from geopy.geocoders import Nominatim
from geopy.distance import great_circle

#We will create a function to geocode city names to coordinates
def geocode_city(city_name):
    geolocator=Nominatim(user_agent="geo_analysis")
    location =geolocator.geocode(city_name)
    return(location.latitude, location.longitude)

#we will create another function to calculate the distance between two points
def distance_calculator(point1, point2):
    return great_circle(point1, point2).miles

#we will now get input from the users
if __name__=="__main__":
    city_name=input("Enter city name to find nearby cities:")
    city_coord =geocode_city(city_name)
    print(f"Coordinates of the {city_name}: {city_coord}")
    big_cities={
        "New Your": (40.7128, -74.0060),
        "Los Angeles":(34.0522, -118.244),
        "Chicago": (41.8781, -87.6298),
        "Detriot":(42.3314, -83.0458),
        "Atlanta": (33.7488, -84.3877),
        "Houston": (29.7604, -95.3698),
        "San Antonio":(29.4241, -98.4936),
        "Dallas": (32.7767, -96.7970)
    }

    #Let's create a dictionary to store nearby cities
    nearby_cities ={}
    for city, coords in big_cities.items():
        distance = distance_calculator(city_coord, coords)
        if distance <= 500:
            nearby_cities[city]= coords

#Now we will create a map and display the nearby cities
nearMap=Map(center=(city_coord[0], city_coord[1]), zoom=6, basemap=basemaps.OpenStreetMap.Mapnik)
#We will add marker for input city
input_city_marker=Marker(location=(city_coord[0], city_coord[1]), title=city_name)
nearMap.add_layer(input_city_marker) #we add the marker to the map

#Now we will add marker for nearby cities
for city, coords in nearby_cities.items():
    near_by_cities_marker=CircleMarker(location=(coords[0], coords[1]), radius=15, color="blue", fill_color="blue", fill_opacity=0.6)
    nearMap.add_layer(near_by_cities_marker) #we add the marker to the map

#We will add layer control to toggle the visibility of the map
nearMap.add_control(LayersControl())

#Finally we will display the map
display(nearMap)

<font size="6">
Analyzing Population Density for Urban Planning

In [ ]:
import geopandas as gpd
import pandas as pd
import folium
from folium.features import GeoJsonTooltip
import branca.colormap as cm
from google.colab import files

In [ ]:
#First Let's upload our pop csv file to google colab
upload_data=files.upload()

Saving usa_population_2019.csv to usa_population_2019 (1).csv


In [ ]:
upload_data=files.upload()

Saving us-states.json to us-states.json


In [ ]:
#we will now load the data using pandas
pop_data=pd.read_csv("usa_population_2019.csv")

In [ ]:
#we will display the first 10 rows of the data
pop_data.head(10)

,Geographic Area,Postal Code,Total Resident Population
0,United States,NaN,328239523
1,Northeast,NaN,55982803
2,Midwest,NaN,68329004
3,South,NaN,125580448
4,West,NaN,78347268
5,Alabama,AL,4903185
6,Alaska,AK,731545
7,Arizona,AZ,7278717
8,Arkansas,AR,3017804
9,California,CA,39512223


In [ ]:
#now lets load the json file using geopandas
gdf=gpd.read_file("us-states.json")

In [ ]:
#Now let's merge the population data with the gdf
merged_data=gdf.merge(pop_data, left_on="name", right_on="Geographic Area")
merged_data.head(10)

,id,name,geometry,Geographic Area,Postal Code,Total Resident Population
0,AL,Alabama,"POLYGON ((-87.3593 35.00118, -85.60668 34.9847...",Alabama,AL,4903185
1,AK,Alaska,"MULTIPOLYGON (((-131.60202 55.11798, -131.5691...",Alaska,AK,731545
2,AZ,Arizona,"POLYGON ((-109.0425 37.00026, -109.04798 31.33...",Arizona,AZ,7278717
3,AR,Arkansas,"POLYGON ((-94.47384 36.50186, -90.15254 36.496...",Arkansas,AR,3017804
4,CA,California,"POLYGON ((-123.23326 42.00619, -122.37885 42.0...",California,CA,39512223
5,CO,Colorado,"POLYGON ((-107.91973 41.00391, -105.72895 40.9...",Colorado,CO,5758736
6,CT,Connecticut,"POLYGON ((-73.05353 42.03905, -71.79931 42.022...",Connecticut,CT,3565287
7,DE,Delaware,"POLYGON ((-75.41409 39.80446, -75.5072 39.6839...",Delaware,DE,973764
8,FL,Florida,"POLYGON ((-85.49714 30.99754, -85.00421 31.003...",Florida,FL,21477737
9,GA,Georgia,"POLYGON ((-83.10919 35.00118, -83.32279 34.787...",Georgia,GA,10617423


In [ ]:
#We will use the geometry field to calculate the land area
#we created a column called area_km2 in the merge_data table and calculate area into it

merged_data["area_km2"] = merged_data["geometry"].to_crs({'init': 'epsg:3395'}).area/10**6

#we will now calculate pop density
merged_data["pop_density"]=merged_data["Total Resident Population"]/merged_data["area_km2"]

print(merged_data[["name", "Total Resident Population", "area_km2", "pop_density"]])

NameError: name 'merged_data' is not defined

Lets Display The population density in interactive map

In [ ]:
min_pop_density=merged_data["pop_density"].min()
max_pop_density=merged_data["pop_density"].max()
color_map = cm.linear.YlOrRd_09.scale(min_pop_density, max_pop_density)

def style_function(feature):
  density=feature["properties"]["pop_density"]
  return {
      "fillColor":color_map(density),
      "color":"black",
      "weight":1,
      "fillOpacity":0.6
  }

#Create Interative Map and add the pop_density and set the color and other properties
m =folium.Map(location=[37.8, -96], zoom_start=4, tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri")
folium.GeoJson(
    merged_data,
    style_function=style_function,
    tooltip=GeoJsonTooltip(fields=["name", "pop_density"],
                           aliases=["State:", "Population Density:"],
                           localize=True)
).add_to(m)

#Add the color scheme legend
color_map.caption = "Population Density"
color_map.add_to(m)

#save as HTML file
m.save("US_Population_Density.html")

#display the map
m

NameError: name 'merged_data' is not defined

## **Monitoring Air Quality**

In [ ]:
import os
import folium
from folium.plugins import HeatMap, HeatMapWithTime
import pandas as pd
import numpy as np
from google.colab import files

In [ ]:
#We upload our data to google colab
uploaded =files.upload()

In [ ]:
#Read the data using pandas
AQ_Data=pd.read_csv("OhioAirQuality_data.csv")
AQ_Data.head(10)

In [ ]:
AQ_Data["County"].value_counts()

,count
County,
Cuyahoga,726
Hamilton,433
Preble,244
Franklin,221


In [ ]:
#Convert date column to day time format and extract week number.
#Because we are monitoring weekly AQI for each loaction
AQ_Data["Date"]=pd.to_datetime(AQ_Data["Date"])
#covert it to week format
AQ_Data["Week"] = AQ_Data["Date"].dt.isocalendar().week
print(AQ_Data['Week'])

In [ ]:
WeeklyMean_AQI= AQ_Data.groupby(["County", "Week"]).agg({
    "Site Latitude": "first",
    "Site Longitude": "first",
    "Daily AQI Value": lambda x: np.mean(x)

}).reset_index()

print("Weekly mean AQI value per Locations:")

#lets loop through each locations
for location in WeeklyMean_AQI["County"].unique():
  location_data=WeeklyMean_AQI[WeeklyMean_AQI["County"]==location]
  print(f"Location: {location}")
  for index, row in location_data.iterrows():
    print(f"Week: {row['Week']}, Average AQI Value: {row['Daily AQI Value']:.2f}")
  print("\n")



In [ ]:
#Now let's plot the map on interactive map
df_week_list=[] #list to store LatLon and AQI values
#let's create a loop to iterate AQI values for each week
for week in range(1, 10):
  weekly_data=WeeklyMean_AQI[WeeklyMean_AQI["Week"]==week][['Site Latitude',	'Site Longitude', 'Daily AQI Value']]
  df_week_list.append(weekly_data.values.tolist())

#now let's initiat folium map centered around Columbus
base_map=folium.Map(location=[39.9622, -83.0006], zoom_start=10, max_zoom=23, control_scale=True,
                     tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri")
#now we will create heatmap with time layer
HeatMapWithTime(df_week_list, radius=70, gradient={0.05:"blue", 0.5:"green", 0.75:"yellow", 1.0:"red"},
                min_opacity=0.5,
                max_opacity=0.8,
                use_local_extrema=True).add_to(base_map)
base_map.save("WeeklyAQI_HeatMap.html")

base_map


# **MAPPING FLOOD RISK**

In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap
from google.colab import files

In [ ]:
upload = files.upload()

Saving FloodAEGISDataset.csv to FloodAEGISDataset.csv


In [ ]:
#Let's read the uploaded data
floodData=pd.read_csv("FloodAEGISDataset.csv")
floodData.head()


,lat,lon,flood_heig,elevation,precipitat
0,14.640394,121.055708,0,54.553295,9.0
1,14.698299,121.002132,0,21.856272,10.0
2,14.698858,121.100261,0,69.322807,16.0
3,14.571310,120.983334,0,10.987241,8.0
4,14.762232,121.075735,0,87.889847,18.0


In [ ]:
#Clean the data. We will convert the columns to numeric type
Numeric_columns=["lat", "lon", "flood_heig", "elevation", "precipitat"]
floodData[Numeric_columns] = floodData[Numeric_columns].apply(pd.to_numeric, errors="coerce")

#drop rows with missing values
floodData_cleaned=floodData.dropna()
#check to confirm if the data does not have missing values
print(f'NaN after cleaning:')
print(floodData_cleaned.isnull().sum())



In [ ]:
#Normalized the data
floodData_cleaned["Normalized_flood_heig"]=(floodData_cleaned["flood_heig"] - floodData_cleaned["flood_heig"].max())/(floodData_cleaned["flood_heig"].max() - floodData_cleaned["flood_heig"].min())
floodData_cleaned["Normalized_elevation"]=(floodData_cleaned["elevation"] - floodData_cleaned["elevation"].max())/(floodData_cleaned["elevation"].max() - floodData_cleaned["elevation"].min())
floodData_cleaned["Normalized_precipitat"]=(floodData_cleaned["precipitat"] - floodData_cleaned["precipitat"].max())/(floodData_cleaned["precipitat"].max() - floodData_cleaned["precipitat"].min())

floodData_cleaned.head()


In [ ]:
#Elevation is inversely proportional to flood risk. Meaning the higher the elevation, the lower the flood risks. So we need to inverse the elevation
floodData_cleaned['Normalized_elevation']=1-floodData_cleaned['Normalized_elevation']

In [ ]:
#Calculate flood risk score for each location. We will apply weights to each variable and calculate the risk score
floodData_cleaned["risk_score"]=(floodData_cleaned['Normalized_flood_heig']*0.4 + floodData_cleaned['Normalized_elevation']*0.3+floodData_cleaned['Normalized_precipitat']*0.3)

In [ ]:
print("\Risk Score for Each Location")
for index, row in floodData_cleaned.iterrows():
  print(f"Latitude:{row['lat']}, Longitude:{row['lon']}, Flood Height:{row['flood_heig']}, Risk Score:{row['risk_score']:.3f}")

In [ ]:
#Now let's map the flood risk using folium

#we first create function to categorize the risk score using colors
def get_color(risk_score):
  if risk_score<0.2:
    return "green"
  elif 0.2<=risk_score<0.4:
    return "yellow"
  elif 0.4<=risk_score<0.6:
    return "orange"
  elif 0.6<=risk_score<0.8:
    return "red"
  else:
    return "dark red"
#we will apply the function to our data
floodData_cleaned["risk_color"] = floodData_cleaned["risk_score"].apply(get_color)

#Now we initialized folium map
manila_coord =[14.5995, 120.9842]
map_flood_risk=folium.Map(location=[14.5995, 120.9842], zoom_start=12, tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri")
heat_data=floodData_cleaned[['lat', 'lon', 'risk_score']].values.tolist()
HeatMap(heat_data, radius=15, max_zoom=13).add_to(map_flood_risk)

print("\nRisk Score for Each Location")
for index, row in floodData_cleaned.iterrows():
  print(f"Latitude:{row['lat']}, Longitude:{row['lon']}, Flood Height:{row['flood_heig']}, Risk Score:{row['risk_score']:.3f}, risk color:{row['risk_color']}")

map_flood_risk.save("Manila_flood_risk.html")
map_flood_risk

# **Mapping Snow Cover**

In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap
from geopy.geocoders import Nominatim
from google.colab import files

In [ ]:
uploaded =files.upload()

In [ ]:
ny_sno_data=pd.read_csv("nyc_Jan_Jun_2016_weat.csv")
ny_sno_data.head()

In [ ]:
#Data Clean
ny_sno_data["SNOW"]=pd.to_numeric(ny_sno_data["SNOW"], errors="coerce")


In [ ]:
#Filter snow for snow that are positive
snow_data_positive=ny_sno_data[ny_sno_data["SNOW"]>0]

In [ ]:
#Filter for dates with highest snow
max_snow_per_location=snow_data_positive.loc[snow_data_positive.groupby(["LATITUDE", "LONGITUDE"])["SNOW"].idxmax()]
print("\nSnow Depth in Inches for Each Location:")
for index, row in max_snow_per_location.iterrows():
  print(f"Latitude:{row['LATITUDE']}, Longitude:{row['LONGITUDE']}, Snow Depth:{row['SNOW']:.2f}")


#Let's geocode the LatLong
geolocator=Nominatim(user_agent="snow_depth locator")
top_3_locations=max_snow_per_location.nlargest(3, 'SNOW')
print("\nTop 3 Locations with Highest Snow Depth:")
for index, row in top_3_locations.iterrows():
  location=geolocator.reverse((row["LATITUDE"], row["LONGITUDE"]), exactly_one=True)
  address=location.address if location else "Address not found"
  print(f"Latitude:{row['LATITUDE']}, Longitude:{row['LONGITUDE']}, Address:{address}, Snow Depth:{row['SNOW']:.2f}")

In [ ]:
#Let's Map the snow depth
heatmap_data=max_snow_per_location[['LATITUDE', 'LONGITUDE', 'SNOW']].values.tolist()
map_center=[40.7128, -74.0060]
map_snow_depth=folium.Map(location=map_center, zoom_start=8, tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri")
HeatMap(heatmap_data, radius=45).add_to(map_snow_depth)
map_snow_depth.save("NYC_snow_depth.html")
map_snow_depth

### **Modeling and Optimizing Routes**

In [ ]:
import folium
import osmnx as ox
import networkx as nx
from ipyleaflet import Map, Polyline, Marker
from ipyleaflet import basemaps

In [ ]:
#Performing the analysis
origin_coord=[41.6578629, -83.6144666]
Destination_Coord=[41.587708, -83.811491]
center = ((origin_coord[0] + Destination_Coord[0]) / 2,
          (origin_coord[1] + Destination_Coord[1]) / 2)
streetnetwort=ox.graph_from_point(center, dist=12000, network_type="drive") #Downloading Streetnetwork
origin_node=ox.distance.nearest_nodes(streetnetwort, origin_coord[1], origin_coord[0])
destination_node=ox.distance.nearest_nodes(streetnetwort, Destination_Coord[1], Destination_Coord[0])
shortest_path=nx.shortest_path(streetnetwort, origin_node, destination_node, weight='length')
path_coords=[(streetnetwort.nodes[node]['y'], streetnetwort.nodes[node]['x']) for node in shortest_path]

#Let's create a map
m =Map(center=[(origin_coord[0]+Destination_Coord[0])/2, (origin_coord[1]+Destination_Coord[1])/2], zoom=12, basemap=basemaps.Esri.WorldStreetMap)
O_marker=Marker(location=origin_coord, draggable=False, title="UToledo Main Campus")
D_marker=Marker(location=Destination_Coord, draggable=False, title="Toledo Airport")
m.add_layer(O_marker)
m.add_layer(D_marker)
path_line=Polyline(locations=path_coords, color="blue", weight=5, fill=False)
m.add_layer(path_line)
m


## **Modelling and Optimizing Bus Route Using Dijkstra Algorithm**

In [ ]:
import heapq
import folium
from folium.plugins import HeatMap

In [ ]:
UToledo =(41.658139, -83.615205)
UTMC= (41.617334, -83.616837)
National_Museum_Great_Lakes =(41.656276, -83.515686)
ProMedica_Toledo_Hospital= (41.674386, -83.595582)
Franklin_Park_Mall= (41.694408, -83.640538)

In [ ]:
graph={
    "Franklin Park Mall":[("University of Toledo", 4.6), ("UTMC", 7.7), ("National Museum of the Great Lakes", 11.3), ("ProMedica_Toledo_Hospital", 3.3)],
    "University of Toledo":[("UTMC", 4.8),("National Museum of the Great Lakes", 6.4),("ProMedica_Toledo_Hospital", 2.8), ("Franklin Park Mall", 4.6)],
    "UTMC":[("University of Toledo", 4.8), ("National Museum of the Great Lakes", 7.2),("ProMedica_Toledo_Hospital", 5.8), ("Franklin Park Mall", 7.7)],
    "ProMedica_Toledo_Hospital":[("University of Toledo", 2.8), ("UTMC", 5.8), ("National Museum of the Great Lakes",9.1), ("Franklin Park Mall", 3.3)],
    "National Museum of the Great Lakes":[("University of Toledo", 6.4), ("UTMC", 7.2), ("ProMedica_Toledo_Hospital",9.1), ("Franklin Park Mall", 11.3)],

}

In [ ]:
def find_shortest_path(graph, nodes):
  import itertools
  all_paths=itertools.permutations(nodes)
  shortest_path=None
  min_distance=float('inf')
  for path in all_paths:
    total_distance=0
    valid_path=True
    for i in range(len(path)-1):
      current_node=path[i]
      next_node=path[i+1]
      find_edge=False
      for neighbor, weight in graph[current_node]:
        if neighbor == next_node:
          total_distance += weight
          found_edge=True
          break
      if not found_edge:
        valid_path=False
        break
    if valid_path and path[0]=="Franklin Park Mall" and path[-1] == "National Museum of the Great Lakes" and total_distance<min_distance:
      min_distance=total_distance
      shortest_path=path
  return shortest_path, min_distance

In [ ]:
#Define the nodes to visit in specify order
nodes_to_visit=["Franklin Park Mall", "University of Toledo", "UTMC", "ProMedica_Toledo_Hospital", "National Museum of the Great Lakes"]
shortest_path, total_distance=find_shortest_path(graph, nodes_to_visit)
print("Shortest Path:", shortest_path)
print("Total Distance:", total_distance, "miles")


#Define coordinates for each node
coordinates={
    "Franklin Park Mall": Franklin_Park_Mall,
    "University of Toledo": UToledo,
    "UTMC": UTMC,
    "ProMedica_Toledo_Hospital": ProMedica_Toledo_Hospital,
    "National Museum of the Great Lakes": National_Museum_Great_Lakes
}

#Create a map using folium
map_center = coordinates["University of Toledo"]
MyMap=folium.Map(location=map_center, zoom_start=12, tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri")
for nodes, coord in coordinates.items():
  folium.Marker(location=coord, popup=nodes).add_to(MyMap)
path_coordinates=[coordinates[node] for node in shortest_path]
folium.PolyLine(locations=path_coordinates, color="blue", weight=5, opacity=0.7).add_to(MyMap)
MyMap.save("Bus_Route_Map.html")
MyMap


## Mapping Real Estate Price

In [ ]:
import folium
import pandas as pd
import numpy as np
from folium.plugins import HeatMap
import branca.colormap as cm

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("subl1m1nal/housing-prices-amsterdam")

print("Path to dataset files:", path)

In [ ]:
file_path="/root/.cache/kagglehub/datasets/subl1m1nal/housing-prices-amsterdam/versions/1/HousingPrices-Amsterdam-August-2021.csv"

In [ ]:
data=pd.read_csv(file_path)
data.head()

In [ ]:
#Clean data
data=data.dropna(subset=["Price", "Lon",	"Lat"])

#Calculate map center based on the average coordinates
map_center=[data["Lat"].mean(), data["Lon"].mean()]

#Create folium map centered on map_center variable
m=folium.Map(location=map_center, zoom_start=12, tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri")

#Create a linear color scheme for pricing
color_map=cm.LinearColormap(
    colors=["#4b0082", "#800080", "#8b0000", "#B22222", "#FF0000"],
    vmin=data["Price"].min(),
    vmax=data["Price"].max(),
    caption="Price"
).add_to(m)

#Loop through each row in the data frame
for row in data.itertuples():
  folium.CircleMarker(
      location=[row.Lat, row.Lon],
      radius=6,
      color=color_map(row.Price),
      fill=True,
      fill_color=color_map(row.Price),
      fill_opacity=0.9,
      weight=0.3,
      popup=f"Price: ${row.Price}"
  ).add_to(m)

m.save("Real_Estate_Price.html")
m

# **Createing 3D Map from OpenStreetMap**

In [ ]:
#import libraries
import pydeck as pdk #for 3D visualization
import osmnx as ox   #for downloading OpenStreetMap data
import pandas as pd #for handling and manipulating data
import re
from shapely.geometry import Polygon, MultiPolygon


#set location
Toledo_Downtown= (41.615241, -83.696913)

#Define tags to fetch only buildings
tags={"building":True}
buildings=ox.features_from_point(Toledo_Downtown, tags=tags, dist=15000)

#Create functions to extract building height values
def to_number(value):
    if value is None or pd.isna(value):
        return None
    match = re.search(r"\d+(\.\d+)?", str(value))
    return float(match.group()) if match else None

def get_buildings(buildings, default_height=10):
    data = []
    for _, row in buildings.iterrows():
        height = to_number(row.get("height"))
        if not height:
            levels = to_number(row.get("building:levels"))
            height = levels * 3 if levels else default_height

        geom = row.geometry
        if isinstance(geom, Polygon):
            parts = [geom]
        elif isinstance(geom, MultiPolygon):
            parts = list(geom.geoms)
        else:
            continue

        for part in parts:
            data.append({"coordinates": list(part.exterior.coords), "height": height})
    return pd.DataFrame(data)


df = get_buildings(buildings)

#create a 3D polygon layer for the building
layer=pdk.Layer(
    "PolygonLayer",
    df,
    get_polygon="coordinates",
    get_elevation="height",
    get_fill_color=[180, 150, 130],
    pickable=True,
    extruded=True
)

#Set initial view of the map
view_state=pdk.ViewState(
    latitude=Toledo_Downtown[0],
    longitude=Toledo_Downtown[1],
    zoom=15,
    pitch=40,
)

#Renter and save the map to html
pdk.Deck(layers=[layer], initial_view_state=view_state).to_html("Toledo_3d_real_heigh.html")

In [ ]:
buildings = buildings[buildings.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()

def estimate_height(row, default_height=10):
    height = to_number(row.get("height"))
    if not height:
        levels = to_number(row.get("building:levels"))
        height = levels * 3 if levels else default_height
    return height

buildings["height_m"] = buildings.apply(estimate_height, axis=1)
buildings[["height_m", "geometry"]].to_file("toledo_buildings.shp")

# **Building Machine Learning Model to predict rent price**

In [ ]:
import pandas as pd
import folium
from sklearn.model_selection import train_test_split #for spliting the data
from sklearn.preprocessing import OneHotEncoder, StandardScaler #for encodingscaling the data
from sklearn.compose import ColumnTransformer #for combining the encoders
from sklearn.pipeline import Pipeline #for creating a pipeline
from sklearn.neighbors import KNeighborsRegressor #for building the model

In [ ]:
#Download Data from kaggle
import kagglehub

# Download latest version
path = kagglehub.dataset_download("azharsaleem/real-estate-goldmine-dubai-uae-rental-market")

print("Path to dataset files:", path)

In [ ]:
file_path="/root/.cache/kagglehub/datasets/azharsaleem/real-estate-goldmine-dubai-uae-rental-market/versions/1/dubai_properties.csv"

In [ ]:
data=pd.read_csv(file_path)
data.head()

In [ ]:
import folium
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor

In [ ]:
#Clean the data: remove missing values
data=data.dropna(subset=[
    "Rent","Beds","Baths","Area_in_sqft","Furnishing","City","Latitude","Longitude"
                         ])

#Define features and target variables
features=["Beds", "Baths", "Area_in_sqft", "Furnishing","City", "Latitude", "Longitude"]
target=["Rent"]

#Extract feature columns from dataset
x=data[features]
y=data[target]

#list categorical and numberical features
categorical_features=["Furnishing", "City"]
numerical_features=["Beds", "Baths", "Area_in_sqft", "Latitude", "Longitude"]

#create preprocessor object
preprocessor=ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numerical_features)
    ]
)

#Create pipeline to combine preprocessing and the model
model=Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("KNN", KNeighborsRegressor(n_neighbors=5))
    ]
)

#Split the data into training and testing sets
x_train, x_test, y_train, y_test=train_test_split(x, y, test_size=0.2, random_state=42)

#Train the model
model.fit(x_train, y_train)

print("modeling training completed:")


#Instruction for user
print("\nEnter property details:")
area=float(input("Area(sqft)"))
beds=int(input("Beds:"))
bath=int(input("Bath:"))
furnishing=str(input("Furnishing Status:"))
city=str(input("City:"))
latitude=float(input("Latitude:"))
longitude=float(input("Longitude:"))

#Create Dataframe from user input
input_data=pd.DataFrame([{
    "Area_in_sqft":area,
    "Beds":beds,
    "Baths":bath,
    "Furnishing":furnishing,
    "City":city,
    "Latitude":latitude,
    "Longitude":longitude
}])

#Predict the rent price using the trained model
predicted_rent=model.predict(input_data)[0][0]
print(f"\nPredicted Rent Price: AED{predicted_rent:.2f}")

#Create interactive map and display the results
m=folium.Map(location=[latitude, longitude], zoom_start=12, tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri")

#create function to assign colors based on rent prince
def get_color(rent):
  if rent<data["Rent"].quantile(0.33):
    return "green"
  elif rent<data["Rent"].quantile(0.66):
    return "orange"
  else:
    return "red"

#Create a sample data
data_sample=data.sample(min(500, len(data)), random_state=42)

#iterate throgh the sample data rows
for _, row in data_sample.iterrows():
  folium.CircleMarker(
      location=[row["Latitude"], row["Longitude"]],
      radius=5,
      color=get_color(row["Rent"]),
      fill=True,
      fill_color=get_color(row["Rent"]),
      fill_opacity=0.7,
      popup=f"Rent: AED{row['Rent']:.0f}"
  ).add_to(m)

#Create marker for user input property
folium.CircleMarker(
    location=[latitude, longitude],
    radius=10,
    color="blue",
    fill=True,
    fill_color="blue",
    fill_opacity=1,
    popup=f"Predicted Rent: AED{predicted_rent:,.2f}",
    tooltip="Your Property"
    ).add_to(m)


m
